In [1]:
from res_opt_core import EnergyModel, Grid, AuctionMarket, plot_battery_operation
from hems_resopt.components.ev import EV
import numpy as np
import pandas as pd
def build_input_dict(
    charger_ids: list[str],
    value: float | dict[str, float],
) -> dict[str, float]:
    """
    Build a per-charger input dictionary from either a scalar or an
    existing dict."""

    if isinstance(value, dict):
            return value                                    # already per-charger
    return {cid: value for cid in charger_ids}         # broadcast scalar → dict

In [2]:
from res_opt_core import EnergyModel, Grid, AuctionMarket, plot_battery_operation
from hems_resopt.components.ev import EV
import numpy as np
import pandas as pd

# ── Session Data ──────────────────────────────────────────────────────────────
easee_sessions = pd.read_parquet(r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\01_raw\filtered_easee_sessions.parquet')
site_data = pd.read_parquet(r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\01_raw\filtered_easee_sites.parquet')
easee_sessions['carConnected']    = pd.to_datetime(easee_sessions['carConnected'], utc=True)
easee_sessions['carDisconnected'] = pd.to_datetime(easee_sessions['carDisconnected'], utc=True)

# ── Sessions per site ID with min/max carConnected ────────────────────────────
sessions_per_site = (
    easee_sessions
    .groupby('site_id')
    .agg(
        session_count = ('site_id',      'size'),
        first_session = ('carConnected', 'min'),
        last_session  = ('carConnected', 'max'),
    )
    .reset_index()
    .sort_values('session_count', ascending=False)
)

sessions_per_site = sessions_per_site.merge(
    site_data[['id']].drop_duplicates(),
    left_on='site_id', right_on='id',
    how='left'
).drop(columns='id')

print(sessions_per_site[['site_id', 'session_count', 'first_session', 'last_session']].to_string(index=False))

# pdf = easee_sessions[easee_sessions['site_id'] == 287032]

# ── Price Data ──────────────────────────────────────────────────────────────
df_dyn_tarif = pd.read_csv(r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\01_raw\home_dynamic_Leistungspreis_CKW_25.csv')
df_dyn_tarif['Timestamp'] = pd.to_datetime(df_dyn_tarif['Timestamp'], dayfirst=True, errors='coerce')
df_dyn_tarif['Timestamp'] = df_dyn_tarif['Timestamp'].dt.tz_localize('Europe/Zurich', ambiguous='infer')
df_dyn_tarif['Timestamp'] = df_dyn_tarif['Timestamp'].dt.tz_convert('UTC')
df_dyn_tarif = df_dyn_tarif.set_index('Timestamp', drop=True).sort_index()

# ── Time Index ─────────────────────────────────────────────────────────────────
start_time = pd.Timestamp('2025-01-01', tz='UTC')
end_time   = pd.Timestamp('2026-01-01', tz='UTC')
idx        = pd.date_range(start=start_time, freq='15min', end=end_time, tz='UTC')

# ── Optimization Dataframe -────────────────────────────────────────────────────
columns      = ['power_min', 'power_max', 'e_in', 'e_out', 'e_cap']
df_ev_inputs = pd.DataFrame(index=idx, columns=columns, dtype=float)
df_ev_inputs.index = pd.to_datetime(df_ev_inputs.index, format='mixed', dayfirst=True)

# ── Assign to df_ev_inputs ─────────────────────────────────────────────────────
df_ev_inputs['dyn_tarif'] = df_dyn_tarif['home dynamic']
df_ev_inputs['dyn_tarif'] = df_ev_inputs['dyn_tarif'] / 100 # From Rp to CHF 
df_ev_inputs['dyn_tarif'] = df_ev_inputs['dyn_tarif'].ffill()

 site_id  session_count             first_session              last_session
  300051           2393 2024-12-31 16:34:08+00:00 2026-04-29 12:56:43+00:00
  275947           1603 2024-12-30 15:13:26+00:00 2026-04-29 20:00:11+00:00
  287032            920 2024-12-31 18:11:07+00:00 2026-04-28 16:13:59+00:00
  405891            900 2024-12-31 22:07:17+00:00 2026-04-29 17:21:52+00:00
  408561            371 2025-01-03 08:12:01+00:00 2026-04-25 21:59:36+00:00
  419588            329 2024-12-31 12:50:05+00:00 2026-04-28 19:18:06+00:00
  328456            251 2025-01-02 09:38:44+00:00 2026-04-29 15:53:58+00:00
  587263            212 2025-01-05 13:40:54+00:00 2026-04-25 12:41:51+00:00
  263264            192 2025-01-01 10:30:08+00:00 2026-04-27 15:33:03+00:00
  226605            184 2025-01-01 14:58:42+00:00 2026-04-27 17:00:12+00:00
  585442            126 2025-01-06 19:22:12+00:00 2026-04-24 15:34:37+00:00
  667659             53 2025-10-14 16:43:56+00:00 2026-04-29 16:56:20+00:00


In [3]:
from hems_resopt.utils.pre_process_ev import (
    prepare_sessions,
    build_connection_df,
    get_power_bounds,
    get_e_in_out_capacity,
)
from hems_resopt.utils.post_process_ev import (
    plot_charger_usage,
    plot_representative_week,
    plot_summary_bars,
    compute_ev_optimization_summary,
    print_summary,
)
from res_opt_core import EnergyModel, Grid, AuctionMarket
from hems_resopt.components.ev import EV
import pandas as pd
import numpy as np

# ── Configuration ──────────────────────────────────────────────────────────────
battery_sizes         = 80
power_max_per_charger = 11
power_nominal         = 200

# Kostenansatz: Nur Energie- und Netzkosten werden betrachtet, alle fixen Abgaben werden nachher dazu gerechnet
ELV_tarif             = 0.11
Netz_standard         = 0.06
peak_power_price_dyn  = 1 # CHF/kW
peak_power_price_stat = 1.5 # CHF/kW
fix_abgaben           = 0.0303 + 0.01 + 0.01036 + 0.0216 # CHF/kW (Fixkosten Netz + Konzessionsabgaben + 1.036 Standby Verbrauch + Transaktionsgebühr)
leistung_pro_kWh      = 0.0587 # CHF/kWh (dieser Betrag wird aktuell verrechnet um peak_power_price + Messgebühr zu decken) 

# ── Pre-align tariff ONCE outside the loop ─────────────────────────────────────
# This is the single source of truth — every site uses the same aligned series.
dyn_tarif_aligned = df_ev_inputs['dyn_tarif']

nan_count = dyn_tarif_aligned.isna().sum()
if nan_count > 0:
    raise ValueError(
        f"dyn_tarif has {nan_count} NaN values after reindex+ffill. "
        f"Check that df_dyn_tarif covers {idx[0]} → {idx[-1]}."
    )
else:
    print(f"✅ dyn_tarif aligned: {len(dyn_tarif_aligned)} slots, 0 NaN.")

# ── Results collector ──────────────────────────────────────────────────────────
results_per_site = {}

# ── Main loop ─────────────────────────────────────────────────────────────────
for siteid in easee_sessions['site_id'].unique():

    print(f"\n{'='*60}")
    print(f"  Running simulation for site: {siteid}")
    print(f"{'='*60}")

    try:
        # ── Filter to this site ────────────────────────────────────────────────
        pdf = easee_sessions[easee_sessions['site_id'] == siteid]

        # ── Step 1 — Prepare sessions ──────────────────────────────────────────
        sessions = prepare_sessions(pdf, start_time, end_time)
        sessions = sessions.drop_duplicates()

        charger_ids  = sessions['chargerId'].unique().tolist()
        battery_dict = build_input_dict(charger_ids, battery_sizes)
        power_dict   = build_input_dict(charger_ids, power_max_per_charger)

        # ── Step 2 — Build connection matrix ──────────────────────────────────
        connection_df = build_connection_df(sessions, idx, charger_ids)

        # ── Fresh DataFrame per site — no shared state ─────────────────────────
        df_ev_inputs_site = pd.DataFrame(index=idx, dtype=float)
        df_ev_inputs_site['dyn_tarif'] = dyn_tarif_aligned  # pre-aligned, no NaN

        # ── Step 3 — Fill power bounds & energy bounds ─────────────────────────
        df_ev_inputs_site[['power_min', 'power_max']] = get_power_bounds(
            connection_df, power_dict
        )

        (
            df_ev_inputs_site[['e_in', 'e_out', 'e_cap']],
            capped_kWh_list,
        ) = get_e_in_out_capacity(
            sessions, connection_df, idx, battery_dict, power_dict
        )

        # ── Sanity checks ──────────────────────────────────────────────────────
        checks = {
            'dyn_tarif NaN'  : df_ev_inputs_site['dyn_tarif'].isna().sum(),
            'e_cap NaN'      : df_ev_inputs_site['e_cap'].isna().sum(),
            'e_in > e_cap'   : (df_ev_inputs_site['e_in'].fillna(0)
                                > df_ev_inputs_site['e_cap'].fillna(0) + 1e-6).sum(),
            'e_out > e_cap'  : (df_ev_inputs_site['e_out'].fillna(0)
                                > df_ev_inputs_site['e_cap'].fillna(0) + 1e-6).sum(),
            'pmin > pmax'    : (df_ev_inputs_site['power_min'].fillna(0)
                                > df_ev_inputs_site['power_max'].fillna(0) + 1e-6).sum(),
        }
        failed_checks = {k: v for k, v in checks.items() if v > 0}
        if failed_checks:
            raise ValueError(f"Pre-solve check failed: {failed_checks}")
        print(f"  ✅ Pre-solve checks passed.")

        # ── Build model ────────────────────────────────────────────────────────
        model = EnergyModel(
            num_steps=len(df_ev_inputs_site.index),
            slot_length="15min",
            solver="highs",
        )

        ev_fleet = EV(
            name="EV_Fleet",
            power_nominal=power_nominal,
            power_min=df_ev_inputs_site['power_min'],
            power_max=df_ev_inputs_site['power_max'],
            energy_capacity=df_ev_inputs_site['e_cap'],
            soc_initial=0.0,
            soc_final=None,
            energy_in_slot_start=df_ev_inputs_site['e_in'],
            energy_out_slot_end=df_ev_inputs_site['e_out'],
        )

        dynamischer_tariff = AuctionMarket(
            name='dynamic_tariff',
            price_curve=df_ev_inputs_site['dyn_tarif'],
        )

        grid = Grid(
            name="grid",
            assets=[ev_fleet],
            markets=[dynamischer_tariff],
        )

        model.add_component(ev_fleet)
        model.add_component(dynamischer_tariff)
        model.add_component(grid)
        model.build_and_run(silent=False)

        df_results = model.results.timeseries_to_pandas()

        # ── Post-processing ────────────────────────────────────────────────────
        df_post_process, summary_dict, monthly_df = compute_ev_optimization_summary(
            df_results=df_results,
            df_ev_inputs=df_ev_inputs_site,
            idx=idx,
            sessions=sessions,
            capped_kWh_list=capped_kWh_list,
            peak_power_price_dyn=peak_power_price_dyn,
            peak_power_price_stat=peak_power_price_stat,
            energy_costs=ELV_tarif,
            fix_costs=fix_abgaben
        )

        print_summary(summary_dict)

        # ── Store results ──────────────────────────────────────────────────────
        results_per_site[siteid] = {
            'df_ev_inputs'    : df_ev_inputs_site,
            'df_post_process' : df_post_process,
            'summary_dict'    : summary_dict,
            'monthly_df'      : monthly_df,
            'sessions'        : sessions,
            'capped_kWh_list' : capped_kWh_list,
            'idx'             : idx,
            'chargers'        : connection_df.columns
        }

    except Exception as e:
        print(f"\n[Site {siteid}] ⛔ FAILED — {type(e).__name__}: {e}")
        results_per_site[siteid] = {
            'error'    : str(e),
            'sessions' : sessions if 'sessions' in dir() else None,
        }
        continue

print("\nAll sites processed. Results stored in `results_per_site`.")

# ── Final status summary ───────────────────────────────────────────────────────
for sid, res in results_per_site.items():
    status = '⛔ FAILED' if 'error' in res else '✅ OK'
    print(f"  Site {sid}: {status}")


INFO: Total kWh removed by power feasibility cap: 26.41 kWh across 7 sessions.


✅ dyn_tarif aligned: 35041 slots, 0 NaN.

  Running simulation for site: 667659
  ✅ Pre-solve checks passed.


c:\Users\ckw-ThCa\AppData\Local\miniconda3\envs\hems_resopt_new\Lib\site-packages\res_opt_core\core\solver\res_opt_highs_solver.py:17: UserWarning: The 'tee' parameter is not supported by the Highs solver and will be set to None.
  warnings.warn("The 'tee' parameter is not supported by the Highs solver and will be set to None.")
INFO: Total kWh removed by power feasibility cap: 215.95 kWh across 14 sessions.



                    EV CHARGING OPTIMISATION — SUMMARY                    

-- Energy --
  Total Energy Charged [kWh]                                 415.31 kWh
  Energy in Raw Sessions [kWh]                               420.25 kWh
  Energy Lost to Power Cap [kWh]                              26.41 kWh
  Energy Lost to Power Cap [%]                                    6.28%

-- Costs (optimized vs flat) --
  Energy Costs Grid Opt [CHF]                                 14.47 CHF
  Energy Costs ELV Opt [CHF]                                  45.68 CHF
  Peak Costs Opt [CHF]                                        33.00 CHF
  Fix Costs (Abgaben etc.) [CHF]                              30.01 CHF
  Total Optimized Costs [CHF]                                123.17 CHF
  Energy Costs Grid (flat tariff) [CHF]                       26.65 CHF
  Energy Costs ELV (flat tariff) [CHF]                        45.68 CHF
  Peak costs (flat tariff) [CHF]                              49.50 CHF
  Fix Costs (

c:\Users\ckw-ThCa\AppData\Local\miniconda3\envs\hems_resopt_new\Lib\site-packages\res_opt_core\core\solver\res_opt_highs_solver.py:17: UserWarning: The 'tee' parameter is not supported by the Highs solver and will be set to None.
  warnings.warn("The 'tee' parameter is not supported by the Highs solver and will be set to None.")
INFO: Total kWh removed by power feasibility cap: 113.97 kWh across 13 sessions.


...


[Site 405891] ⛔ FAILED — InfeasibleOrUnboundedException: Solver status infeasible: A feasible solution was not found, so no solution can be loaded. Please set opt.config.load_solutions=False and check results.solution_status and results.incumbent_objective before loading a solution.

  Running simulation for site: 275947


INFO: Total kWh removed by power feasibility cap: 288.33 kWh across 71 sessions.


  ✅ Pre-solve checks passed.


c:\Users\ckw-ThCa\AppData\Local\miniconda3\envs\hems_resopt_new\Lib\site-packages\res_opt_core\core\solver\res_opt_highs_solver.py:17: UserWarning: The 'tee' parameter is not supported by the Highs solver and will be set to None.
  warnings.warn("The 'tee' parameter is not supported by the Highs solver and will be set to None.")



                    EV CHARGING OPTIMISATION — SUMMARY                    

-- Energy --
  Total Energy Charged [kWh]                              31,039.78 kWh
  Energy in Raw Sessions [kWh]                            29,552.83 kWh
  Energy Lost to Power Cap [kWh]                             288.33 kWh
  Energy Lost to Power Cap [%]                                    0.98%

-- Costs (optimized vs flat) --
  Energy Costs Grid Opt [CHF]                                638.78 CHF
  Energy Costs ELV Opt [CHF]                                  3,414 CHF
  Peak Costs Opt [CHF]                                       627.00 CHF
  Fix Costs (Abgaben etc.) [CHF]                              2,243 CHF
  Total Optimized Costs [CHF]                                 6,923 CHF
  Energy Costs Grid (flat tariff) [CHF]                       1,992 CHF
  Energy Costs ELV (flat tariff) [CHF]                        3,414 CHF
  Peak costs (flat tariff) [CHF]                             940.50 CHF
  Fix Costs (

In [4]:
# ── Inspect chargers per site from stored results ─────────────────────────────
for siteid, res in results_per_site.items():
    if 'error' not in res:
        chargers = res['sessions']['chargerId'].unique().tolist()
        print(f"  Site {siteid:>8} — {len(chargers):>2} charger(s) : {chargers}")
    else:
        print(f"  Site {siteid:>8} — ⛔ FAILED")


  Site   667659 —  1 charger(s) : ['ECZRY4K4']
  Site   263264 —  1 charger(s) : ['EC5YJDY4']
  Site   226605 — ⛔ FAILED
  Site   585442 —  2 charger(s) : ['EC4ZH6D2', 'ECGFJHBC']
  Site   587263 —  2 charger(s) : ['ECJ6FMAP', 'EC6JBLHT']
  Site   408561 — ⛔ FAILED
  Site   419588 — ⛔ FAILED
  Site   328456 —  2 charger(s) : ['ECWCNW3E', 'EC2TZQC9']
  Site   300051 — 13 charger(s) : ['ECHAARJ4', 'ECEGT94Y', 'ECQM8FKB', 'ECVQ869B', 'ECG8CGJ7', 'ECL44WJ5', 'EC9ZJNQ2', 'ECKXQGW8', 'ECWUQPLA', 'ECZG7JSJ', 'ECP4UNDK', 'ECDTP2N2', 'ECKLF4XE']
  Site   287032 —  7 charger(s) : ['EH44Z9YS', 'ECRCQ5SA', 'ECRARAU6', 'ECW83H8V', 'ECXX7FJM', 'ECMCZPM8', 'ECA98L9Y']
  Site   405891 — ⛔ FAILED
  Site   275947 —  8 charger(s) : ['ECWAZS32', 'EC6GCXL5', 'EC5667C7', 'ECJ8C2JP', 'ECU4SPAU', 'ECND4RB6', 'EC8WLUXQ', 'ECJ3J6CD']


In [5]:
import os
import pandas as pd

# ── Output paths ───────────────────────────────────────────────────────────────
output_dir = r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\03_processed'
os.makedirs(output_dir, exist_ok=True)

csv_path    = os.path.join(output_dir, 'summary_all_sites_New_EV_Class.csv')
excel_path  = os.path.join(output_dir, 'summary_all_sites_New_EV_Class.xlsx')
monthly_csv = os.path.join(output_dir, 'monthly_all_sites_New_EV_Class.csv')

# ── Helper: strip timezone from all datetime columns + index ──────────────────
def strip_tz(df: pd.DataFrame) -> pd.DataFrame:
    """Remove timezone info from index, datetime columns, and datetime values in object columns."""
    df = df.copy()

    # ── Strip index timezone ──────────────────────────────────────────────────
    if hasattr(df.index, 'tz') and df.index.tz is not None:
        df.index = df.index.tz_localize(None)

    for col in df.columns:
        # ── Proper datetime columns (dtype = datetime64[ns, tz]) ──────────────
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            if hasattr(df[col].dt, 'tz') and df[col].dt.tz is not None:
                df[col] = df[col].dt.tz_localize(None)

        # ── Object columns that may contain tz-aware Timestamp values ─────────
        # (happens after .T — datetime values become plain Python objects)
        elif df[col].dtype == object:
            df[col] = df[col].apply(
                lambda v: v.tz_localize(None)          # already a Timestamp → strip tz
                if isinstance(v, pd.Timestamp) and v.tzinfo is not None
                else v
            )

    return df

# ── Build summary_dict rows (one row per site) ────────────────────────────────
rows = []
for siteid, res in results_per_site.items():
    if 'error' not in res:
        row = {'site_id': siteid, 'status': '✅ OK'}
        row.update(res['summary_dict'])
    else:
        row = {'site_id': siteid, 'status': f"⛔ FAILED — {res['error']}"}
    rows.append(row)

df_summary_all = (
    pd.DataFrame(rows)
    .set_index('site_id')
    .sort_index()
).T   # metrics as rows, sites as columns

# ── Build monthly_df — one row per (site × month) ─────────────────────────────
monthly_rows = []
for siteid, res in results_per_site.items():
    if 'error' not in res and res.get('monthly_df') is not None:
        df_m = res['monthly_df'].copy()
        df_m.insert(0, 'site_id', siteid)
        monthly_rows.append(df_m)

if monthly_rows:
    df_monthly_all = pd.concat(monthly_rows, axis=0).reset_index()
    if 'index' in df_monthly_all.columns:
        df_monthly_all = df_monthly_all.rename(columns={'index': 'month'})
else:
    df_monthly_all = pd.DataFrame()
    print("⚠️  No monthly_df data found in results_per_site.")

# ── Strip timezones before Excel export ───────────────────────────────────────
df_summary_excel = strip_tz(df_summary_all)
df_monthly_excel = strip_tz(df_monthly_all)

print(f"df_monthly_all shape : {df_monthly_excel.shape}")

# ── Save monthly CSV ──────────────────────────────────────────────────────────
df_monthly_all.to_csv(monthly_csv, encoding='utf-8-sig', float_format='%.4f', index=False)
print(f"✅ Monthly CSV saved to:\n   {monthly_csv}")

# ── Save Excel — two sheets ───────────────────────────────────────────────────
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

    # ── Sheet 1: Summary (transposed, tz-stripped) ────────────────────────────
    df_summary_excel.to_excel(writer, sheet_name='Summary per Site', index=True)
    ws1 = writer.sheets['Summary per Site']
    for col in ws1.columns:
        max_len = max(
            len(str(cell.value)) if cell.value is not None else 0
            for cell in col
        )
        ws1.column_dimensions[col[0].column_letter].width = min(max_len + 4, 50)
    ws1.freeze_panes = 'B2'

    # ── Sheet 2: Monthly values (tz-stripped) ─────────────────────────────────
    if not df_monthly_excel.empty:
        df_monthly_excel.to_excel(writer, sheet_name='Monthly per Site', index=False)
        ws2 = writer.sheets['Monthly per Site']
        for col in ws2.columns:
            max_len = max(
                len(str(cell.value)) if cell.value is not None else 0
                for cell in col
            )
            ws2.column_dimensions[col[0].column_letter].width = min(max_len + 4, 50)
        ws2.freeze_panes = 'A2'

print(f"✅ Excel saved to:\n   {excel_path}")

# ── Save summary CSV ──────────────────────────────────────────────────────────
df_summary_all.to_csv(csv_path, encoding='utf-8-sig', float_format='%.4f')
print(f"✅ Summary CSV saved to:\n   {csv_path}")

# ── Preview ───────────────────────────────────────────────────────────────────
print(f"\n📊 Summary shape : {df_summary_excel.shape}")
print(f"📊 Monthly shape : {df_monthly_excel.shape}")
print(df_monthly_excel.head(10).to_string(index=False))


df_monthly_all shape : (104, 13)
✅ Monthly CSV saved to:
   C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\03_processed\monthly_all_sites_New_EV_Class.csv
✅ Excel saved to:
   C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\03_processed\summary_all_sites_New_EV_Class.xlsx
✅ Summary CSV saved to:
   C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\03_processed\summary_all_sites_New_EV_Class.csv

📊 Summary shape : (31, 12)
📊 Monthly shape : (104, 13)
     month  site_id  energy_charged_kWh  energy_costs_grid_opt_CHF  energy_costs_grid_stand_CHF  energy_costs_ELV_opt_CHF  energy_costs_ELV_stand_CHF  peak_kW  peak_costs_opt_CHF  peak_costs_stand_CHF  fix_costs_CHF  total_costs_optimized_CHF  total_costs_standard_CHF
2025-01-31   667659            0.000000                   0.000000                     0.000000                  0.000000 